In [ ]:
# ============================================================
# Q3 - IMDb Dataset - Complete NLP Preprocessing Pipeline
# 
# ============================================================

import pandas as pd
import re
import nltk
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer

# Download required NLTK resources
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ------------------------------------------------------------
# Load IMDb dataset
# ------------------------------------------------------------
file_path = "/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nOriginal columns:", df.columns.tolist())
print("\nSample review:")
print(df["review"].iloc[0][:500])


# ============================================================
# (a) Lowercase + Regex Cleaning
# ============================================================

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Keep only alphabetic characters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["cleaned_text"] = df["review"].apply(clean_text)

print("\n" + "="*70)
print("(a) CLEANING EXAMPLE")
print("="*70)

print("\nOriginal:")
print(df["review"].iloc[0][:500])

print("\nAfter regex cleaning:")
print(df["cleaned_text"].iloc[0][:500])


# ============================================================
# (b) Tokenization + Stop Word Removal
# ============================================================

stop_words = set(stopwords.words("english"))

def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)

    # Keep alphabetic tokens and remove stopwords
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return tokens


df["tokens"] = df["cleaned_text"].apply(tokenize_and_remove_stopwords)

print("\n" + "="*70)
print("(b) TOKENIZATION + STOP WORD REMOVAL")
print("="*70)

print("\nTokens from first review:")
print(df["tokens"].iloc[0][:50])


# ============================================================
# (c) Lemmatization
# ============================================================

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]


df["lemmatized_tokens"] = df["tokens"].apply(lemmatize_tokens)

# Unique tokens BEFORE lemmatization
unique_before = set(
    word
    for tokens in df["tokens"]
    for word in tokens
)

# Unique tokens AFTER lemmatization
unique_after = set(
    word
    for tokens in df["lemmatized_tokens"]
    for word in tokens
)

print("\n" + "="*70)
print("(c) LEMMATIZATION")
print("="*70)

print("Unique tokens before lemmatization :", len(unique_before))
print("Unique tokens after lemmatization  :", len(unique_after))
print("Reduction in vocabulary            :", 
      len(unique_before) - len(unique_after))

# Show examples where different words become the same lemma
lemma_examples = {}

for word in unique_before:
    lemma = lemmatizer.lemmatize(word)

    if word != lemma:
        lemma_examples.setdefault(lemma, []).append(word)

print("\nExamples of changes after lemmatization:")
count = 0

for lemma, words in lemma_examples.items():
    print(f"{words[:5]} -> {lemma}")
    count += 1

    if count >= 10:
        break


# Create final preprocessed text
df["preprocessed_text"] = df["lemmatized_tokens"].apply(
    lambda tokens: " ".join(tokens)
)


# ============================================================
# (d) Own word-frequency feature function
# ============================================================

def extract_word_features(review):
    """
    Returns a dictionary containing the frequency
    of each word in the supplied review.
    """
    tokens = word_tokenize(review.lower())

    # Remove punctuation / irrelevant tokens
    tokens = [
        word for word in tokens
        if word.isalpha()
    ]

    return dict(Counter(tokens))


print("\n" + "="*70)
print("(d) CUSTOM WORD FEATURE EXTRACTION")
print("="*70)

sample_review = "This movie was absolutely amazing and entertaining"

print("\nReview:")
print(sample_review)

print("\nFeature dictionary:")
print(extract_word_features(sample_review))


# ============================================================
# (e) Apply function to given reviews
# ============================================================

review_1 = "This movie was absolutely amazing and entertaining"
review_2 = "The movie was boring and predictable"

features_1 = extract_word_features(review_1)
features_2 = extract_word_features(review_2)

print("\n" + "="*70)
print("(e) FEATURE DICTIONARIES")
print("="*70)

print("\nReview 1:")
print(review_1)
print("Feature dictionary:")
print(features_1)

print("\nReview 2:")
print(review_2)
print("Feature dictionary:")
print(features_2)


# ============================================================
# (f) Bag-of-Words using CountVectorizer
# ============================================================

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(df["preprocessed_text"])

bow_features = vectorizer.get_feature_names_out()

print("\n" + "="*70)
print("(f) BAG-OF-WORDS REPRESENTATION")
print("="*70)

print("Number of documents       :", X_bow.shape[0])
print("Number of unique features :", X_bow.shape[1])
print("Document-Term Matrix shape:", X_bow.shape)


# ============================================================
# (g) Vocabulary before vs after preprocessing
# ============================================================

# Vocabulary from original reviews
original_vocabulary = set()

for review in df["review"]:
    words = re.findall(r"[a-zA-Z]+", review.lower())
    original_vocabulary.update(words)

# Vocabulary after preprocessing
processed_vocabulary = set(bow_features)

# Words that disappeared
removed_words = original_vocabulary - processed_vocabulary

print("\n" + "="*70)
print("(g) VOCABULARY COMPARISON")
print("="*70)

print("Unique vocabulary BEFORE preprocessing :", len(original_vocabulary))
print("Unique vocabulary AFTER preprocessing  :", len(processed_vocabulary))
print("Words removed                          :", len(removed_words))

print("\nSample words that disappeared:")
print(sorted(list(removed_words))[:100])


# ------------------------------------------------------------
# Explain why some words disappeared
# ------------------------------------------------------------

print("\n" + "-"*70)
print("WHY WERE THESE WORDS REMOVED?")
print("-"*70)

# Stopwords removed
removed_stopwords = [
    word for word in removed_words
    if word in stop_words
]

print("\n1. Stop words removed:")
print(removed_stopwords[:30])

# Words that were only punctuation / numbers / irrelevant characters
print("\n2. HTML tags, URLs, punctuation and numbers:")
print("These were removed by the regular-expression cleaning step.")

# Words affected by lemmatization
lemmatized_disappear = []

for word in unique_before:
    lemma = lemmatizer.lemmatize(word)

    if word != lemma:
        lemmatized_disappear.append((word, lemma))

print("\n3. Words changed by lemmatization:")
print(lemmatized_disappear[:30])


# ============================================================
# Final Summary
# ============================================================

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"""
(a) Lowercasing + regex cleaning       : Completed
(b) Tokenization + stopword removal    : Completed
(c) Lemmatization                      : Completed
(d) Custom word-frequency function     : Completed
(e) Two feature dictionaries           : Displayed
(f) Bag-of-Words                       : Completed

Documents                            : {X_bow.shape[0]}
Unique BoW features                  : {X_bow.shape[1]}
Document-Term Matrix                 : {X_bow.shape}

(g) Vocabulary before preprocessing  : {len(original_vocabulary)}
    Vocabulary after preprocessing   : {len(processed_vocabulary)}
    Vocabulary removed               : {len(removed_words)}
""")